In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkPractice").getOrCreate()

employee_data = [
    (101, "Sravan", "Data Engineer", "IT", 75000, "Hyderabad", 28, "2021-05-10", "Male"),
    (102, "Ravi", "Software Engineer", "IT", 68000, "Bangalore", 30, "2020-03-15", "Male"),
    (103, "Priya", "Data Analyst", "Analytics", 62000, "Chennai", 26, "2022-01-12", "Female"),
    (104, "Kiran", "Manager", "HR", 90000, "Mumbai", 35, "2018-07-19", "Male"),
    (105, "Anjali", "HR Executive", "HR", 45000, "Pune", 24, "2023-02-20", "Female"),
    (106, "Vikram", "Data Scientist", "Analytics", 98000, "Delhi", 32, "2019-11-25", "Male"),
    (107, "Sneha", "Developer", "IT", 71000, "Hyderabad", 27, "2021-08-17", "Female"),
    (108, "Rahul", "Tester", "QA", 55000, "Chennai", 29, "2020-06-10", "Male"),
    (109, "Meena", "QA Lead", "QA", 83000, "Bangalore", 33, "2017-09-14", "Female"),
    (110, "Arjun", "Support Engineer", "Support", 50000, "Pune", 31, "2022-04-11", "Male")
]

columns = ["emp_id","name","designation","department","salary","city","age","joining_date","gender"]

emp_df = spark.createDataFrame(employee_data, columns)

df = emp_df

In [0]:
# Top 3 Highest Salaries Department-wise
window_spec = Window.partitionBy("department") \
                    .orderBy(col("salary").desc())

top3_df = df.withColumn(
    "rank",
    row_number().over(window_spec)
)

top3_df.filter(
    col("rank") <= 3
).show()

+------+------+-----------------+----------+------+---------+---+------------+------+----+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|rank|
+------+------+-----------------+----------+------+---------+---+------------+------+----+
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|   1|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|   2|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|   1|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|   2|
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|   1|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|   2|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|   3|
|   109| Meena|          QA Lead|        QA| 83000|Bangalore| 33|  2017-09-14|Female|   1|

In [0]:
# Average Salary City-wise
df.groupBy("city") \
  .agg(
      avg("salary").alias("average_salary")
  ) \
  .show()

+---------+--------------+
|     city|average_salary|
+---------+--------------+
|Hyderabad|       73000.0|
|Bangalore|       75500.0|
|  Chennai|       58500.0|
|   Mumbai|       90000.0|
|     Pune|       47500.0|
|    Delhi|       98000.0|
+---------+--------------+



In [0]:
# Salary Bands
df.withColumn(
    "salary_band",
    when(col("salary") >= 90000, "High Salary")
    .when(col("salary") >= 60000, "Medium Salary")
    .otherwise("Low Salary")
).show()

+------+------+-----------------+----------+------+---------+---+------------+------+-------------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|  salary_band|
+------+------+-----------------+----------+------+---------+---+------------+------+-------------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|Medium Salary|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|Medium Salary|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|Medium Salary|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|  High Salary|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|   Low Salary|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|  High Salary|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|Medium Salary|


In [0]:
# Yearly Salary Report
df.withColumn(
    "year",
    year(to_date(col("joining_date")))
) \
.groupBy("year") \
.agg(
    sum("salary").alias("total_salary")
) \
.show()

+----+------------+
|year|total_salary|
+----+------------+
|2021|      146000|
|2020|      123000|
|2022|      112000|
|2018|       90000|
|2023|       45000|
|2019|       98000|
|2017|       83000|
+----+------------+



In [0]:
# Employees Earning Above Department Average
dept_avg = df.groupBy("department") \
             .agg(
                 avg("salary").alias("avg_salary")
             )

df.join(
    dept_avg,
    "department"
).filter(
    col("salary") > col("avg_salary")
).show()

+----------+------+------+--------------+------+---------+---+------------+------+-----------------+
|department|emp_id|  name|   designation|salary|     city|age|joining_date|gender|       avg_salary|
+----------+------+------+--------------+------+---------+---+------------+------+-----------------+
|        IT|   101|Sravan| Data Engineer| 75000|Hyderabad| 28|  2021-05-10|  Male|71333.33333333333|
|        HR|   104| Kiran|       Manager| 90000|   Mumbai| 35|  2018-07-19|  Male|          67500.0|
| Analytics|   106|Vikram|Data Scientist| 98000|    Delhi| 32|  2019-11-25|  Male|          80000.0|
|        QA|   109| Meena|       QA Lead| 83000|Bangalore| 33|  2017-09-14|Female|          69000.0|
+----------+------+------+--------------+------+---------+---+------------+------+-----------------+



In [0]:
# Highest Paid Employee in Each Department
window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").desc()
)

df.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") == 1
).show()

+------+------+----------------+----------+------+---------+---+------------+------+----+
|emp_id|  name|     designation|department|salary|     city|age|joining_date|gender|rank|
+------+------+----------------+----------+------+---------+---+------------+------+----+
|   106|Vikram|  Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|   1|
|   104| Kiran|         Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|   1|
|   101|Sravan|   Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|   1|
|   109| Meena|         QA Lead|        QA| 83000|Bangalore| 33|  2017-09-14|Female|   1|
|   110| Arjun|Support Engineer|   Support| 50000|     Pune| 31|  2022-04-11|  Male|   1|
+------+------+----------------+----------+------+---------+---+------------+------+----+



In [0]:
# Lowest Paid Employee in Each Department
window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").asc()
)

df.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") == 1
).show()

+------+------+-----------------+----------+------+---------+---+------------+------+----+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|rank|
+------+------+-----------------+----------+------+---------+---+------------+------+----+
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|   1|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|   1|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|   1|
|   108| Rahul|           Tester|        QA| 55000|  Chennai| 29|  2020-06-10|  Male|   1|
|   110| Arjun| Support Engineer|   Support| 50000|     Pune| 31|  2022-04-11|  Male|   1|
+------+------+-----------------+----------+------+---------+---+------------+------+----+



In [0]:
# Total Salary Department-wise
df.groupBy("department") \
  .agg(
      sum("salary").alias("total_salary")
  ) \
  .show()

+----------+------------+
|department|total_salary|
+----------+------------+
|        IT|      214000|
| Analytics|      160000|
|        HR|      135000|
|        QA|      138000|
|   Support|       50000|
+----------+------------+



In [0]:
# Average Salary Department-wise
df.groupBy("department") \
  .agg(
      avg("salary").alias("average_salary")
  ) \
  .show()

+----------+-----------------+
|department|   average_salary|
+----------+-----------------+
|        IT|71333.33333333333|
| Analytics|          80000.0|
|        HR|          67500.0|
|        QA|          69000.0|
|   Support|          50000.0|
+----------+-----------------+



In [0]:
# Department-wise Employee Count
df.groupBy("department") \
  .count() \
  .show()

+----------+-----+
|department|count|
+----------+-----+
|        IT|    3|
| Analytics|    2|
|        HR|    2|
|        QA|    2|
|   Support|    1|
+----------+-----+

